# GPS Tree Geolocation with Gimbal-Based North Alignment

In [ ]:

import os
import re
import math
import cv2
import numpy as np
from typing import Tuple, Optional, Dict
from PIL import Image, ExifTags

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Geolocation"

# Paths (no need to edit)
PAIRS_DIR = f"{BASE_DIR}/Pairs + EXIF Data"
OUTPUT_DIR = f"{BASE_DIR}/Results_gimbal_aligned"
GIMBAL_DATA_FILE = f"{PAIRS_DIR}/GIMBAL_SUMMARY_20251002_212839.txt"

# Constants
EARTH_RADIUS_M = 6378137.0
DEG_TO_RAD = math.pi / 180.0
ALTITUDE = 100.0
FOV = 82.1

# Tilt corrections
EAST_CORRECTION = 0
NORTH_CORRECTION = 0

# ==================== GIMBAL DATA PARSING ====================
def parse_gimbal_data(gimbal_file_path: str) -> Dict[str, float]:
    """Parse gimbal data file and extract yaw angles for each image."""
    gimbal_data = {}
    
    with open(gimbal_file_path, 'r') as f:
        lines = f.readlines()
    
    current_image = None
    for line in lines:
        line = line.strip()
        
        # Check for image name (ends with _other or _ref)
        if line and ':' in line and ('_other:' in line or '_ref:' in line):
            # Extract just the filename without the colon
            current_image = line.replace(':', '')
        
        # Extract Gimbal Yaw value
        elif current_image and 'Gimbal Yaw:' in line and 'DJI Gimbal' not in line:
            # Parse the yaw value
            yaw_str = line.split('Gimbal Yaw:')[1].strip()
            yaw_value = float(yaw_str)
            gimbal_data[current_image] = yaw_value
    
    return gimbal_data

def get_rotation_for_north(gimbal_yaw: float) -> float:
    """
    Calculate rotation angle needed to make image face north.
    Gimbal yaw is the angle from north (clockwise positive).
    Simply rotate the opposite direction by the same amount.
    """
    return -gimbal_yaw

# ==================== IMAGE AND BOX ROTATION ====================
def rotate_image_and_box(image: np.ndarray, cx: float, cy: float, w: float, h: float, angle: float):
    """Rotate both image and YOLO box together. Handles arbitrary angles."""
    orig_H, orig_W = image.shape[:2]
    
    if abs(angle) < 0.01:  # Effectively 0
        return image, cx, cy, w, h, orig_W, orig_H
    
    # For arbitrary angles - just rotate by the given angle
    center = (orig_W // 2, orig_H // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    
    cos = abs(M[0, 0])
    sin = abs(M[0, 1])
    new_W = int(orig_H * sin + orig_W * cos)
    new_H = int(orig_H * cos + orig_W * sin)
    
    M[0, 2] += new_W / 2 - center[0]
    M[1, 2] += new_H / 2 - center[1]
    
    rotated = cv2.warpAffine(image, M, (new_W, new_H))
    
    # Rotate the bounding box center
    box_center = np.array([cx, cy, 1])
    new_center = M @ box_center
    new_cx = new_center[0]
    new_cy = new_center[1]
    
    # Adjust box dimensions
    angle_rad = math.radians(-angle)
    cos_a = abs(math.cos(angle_rad))
    sin_a = abs(math.sin(angle_rad))
    new_w = w * cos_a + h * sin_a
    new_h = w * sin_a + h * cos_a
    
    return rotated, new_cx, new_cy, new_w, new_h, new_W, new_H

# ==================== EXIF FUNCTIONS ====================
def ratio_to_float(v):
    try:
        return float(v.num) / float(v.den)
    except:
        if isinstance(v, tuple) and len(v) == 2:
            return float(v[0]) / float(v[1])
        return float(v)

def dms_to_decimal(dms, ref) -> float:
    d = ratio_to_float(dms[0])
    m = ratio_to_float(dms[1])
    s = ratio_to_float(dms[2])
    dec = d + m/60.0 + s/3600.0
    if ref in [b'S','S',b'W','W']:
        dec = -dec
    return dec

def extract_exif_gps_alt(image_path: str):
    with Image.open(image_path) as im:
        W, H = im.size
        exif = im.getexif()
        if not exif:
            return W, H, None, None, ALTITUDE

        gps_ifd = None
        try:
            gps_ifd = exif.get_ifd(0x8825)
        except:
            pass
        if gps_ifd is None:
            emap = {ExifTags.TAGS.get(k, k): v for k, v in exif.items()}
            maybe = emap.get('GPSInfo')
            if isinstance(maybe, dict):
                gps_ifd = maybe
        if not gps_ifd:
            return W, H, None, None, ALTITUDE

        lat_val = gps_ifd.get(2) or gps_ifd.get('GPSLatitude')
        lat_ref = gps_ifd.get(1) or gps_ifd.get('GPSLatitudeRef')
        lon_val = gps_ifd.get(4) or gps_ifd.get('GPSLongitude')
        lon_ref = gps_ifd.get(3) or gps_ifd.get('GPSLongitudeRef')

        lat = dms_to_decimal(lat_val, lat_ref) if (lat_val and lat_ref) else None
        lon = dms_to_decimal(lon_val, lon_ref) if (lon_val and lon_ref) else None
        
        return W, H, lat, lon, ALTITUDE

# ==================== GPS CONVERSION ====================
def gps_to_meters(lat1: float, lon1: float, lat2: float, lon2: float):
    lat1r = lat1 * DEG_TO_RAD
    lat2r = lat2 * DEG_TO_RAD
    lon1r = lon1 * DEG_TO_RAD
    lon2r = lon2 * DEG_TO_RAD

    dlat = lat2r - lat1r
    dlon = lon2r - lon1r
    lat_avg = 0.5 * (lat1r + lat2r)

    dN = dlat * EARTH_RADIUS_M
    dE = dlon * EARTH_RADIUS_M * math.cos(lat_avg)
    return dE, dN

def meters_to_gps(base_lat: float, base_lon: float, 
                  east_m: float, north_m: float) -> Tuple[float, float]:
    lat_offset = north_m / EARTH_RADIUS_M * (180.0 / math.pi)
    lon_offset = east_m / (EARTH_RADIUS_M * math.cos(base_lat * DEG_TO_RAD)) * (180.0 / math.pi)
    
    return base_lat + lat_offset, base_lon + lon_offset

def pixel_to_meters(u_px: float, v_px: float,
                   W: int, H: int,
                   altitude: float, fovx: float,
                   apply_correction: bool = True) -> Tuple[float, float]:
    fx = (W / 2.0) / math.tan(0.5 * fovx * DEG_TO_RAD)
    fy = fx
    
    cx = W / 2.0
    cy = H / 2.0
    
    dx_px = u_px - cx
    dy_px = v_px - cy
    
    east_m = (dx_px / fx) * altitude
    north_m = -(dy_px / fy) * altitude
    
    if apply_correction:
        east_m += EAST_CORRECTION
        north_m += NORTH_CORRECTION
    
    return east_m, north_m

def meters_to_pixel(east_m: float, north_m: float,
                   W: int, H: int,
                   altitude: float, fovx: float) -> Tuple[float, float]:
    fx = (W / 2.0) / math.tan(0.5 * fovx * DEG_TO_RAD)
    fy = fx
    
    cx = W / 2.0
    cy = H / 2.0
    
    u_px = cx + (east_m / altitude) * fx
    v_px = cy - (north_m / altitude) * fy
    
    return u_px, v_px

def read_yolo_center_px(label_path: str, W: int, H: int):
    with open(label_path, 'r') as f:
        line = f.readline().strip()
    parts = line.split()
    if len(parts) < 5:
        raise ValueError("Invalid YOLO format")
    
    cx, cy, ww, hh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
    return cx * W, cy * H, ww * W, hh * H

# ==================== FILE HANDLING ====================
def find_files_in_pair(pair_folder: str):
    """Find image and label files, properly handling 'other' and 'ref' naming."""
    files = os.listdir(pair_folder)
    
    # Find images
    images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    # Separate other and ref images
    other_img = None
    ref_img = None
    
    for img in images:
        if 'other' in img.lower():
            other_img = img
        elif 'ref' in img.lower():
            ref_img = img
    
    if not other_img or not ref_img:
        print(f"Warning: Could not find both 'other' and 'ref' images in {pair_folder}")
        return None, None, None, None
    
    # Find corresponding label files
    other_base = os.path.splitext(other_img)[0]
    ref_base = os.path.splitext(ref_img)[0]
    
    other_label = other_base + '.txt'
    ref_label = ref_base + '.txt'
    
    # Check if label files exist
    other_label_path = os.path.join(pair_folder, other_label) if other_label in files else None
    ref_label_path = os.path.join(pair_folder, ref_label) if ref_label in files else None
    
    return (
        os.path.join(pair_folder, other_img),
        os.path.join(pair_folder, ref_img),
        other_label_path,
        ref_label_path
    )

# ==================== PROCESS PAIR WITH GIMBAL DATA ====================
def process_pair_with_gimbal(pair_folder: str, gimbal_data: Dict[str, float]) -> Dict:
    """Process a pair using gimbal data for rotation."""
    
    pair_name = os.path.basename(pair_folder)
    print(f"\nProcessing {pair_name} with gimbal alignment...")
    
    # Get files
    other_img_path, ref_img_path, other_lbl_path, ref_lbl_path = find_files_in_pair(pair_folder)
    
    if not other_img_path or not ref_img_path:
        print(f"Skipping {pair_name}: missing images")
        return None
    
    if not other_lbl_path:
        print(f"Skipping {pair_name}: no detection label for 'other' image")
        return None
    
    # Get gimbal data for both images
    other_filename = os.path.splitext(os.path.basename(other_img_path))[0]
    ref_filename = os.path.splitext(os.path.basename(ref_img_path))[0]
    
    other_gimbal_yaw = gimbal_data.get(other_filename, None)
    ref_gimbal_yaw = gimbal_data.get(ref_filename, None)
    
    if other_gimbal_yaw is None or ref_gimbal_yaw is None:
        print(f"Warning: No gimbal data found for {pair_name}")
        print(f"  Looking for: {other_filename} and {ref_filename}")
        return None
    
    # Calculate rotation angles for north alignment
    other_rotation = get_rotation_for_north(other_gimbal_yaw)
    ref_rotation = get_rotation_for_north(ref_gimbal_yaw)
    
    print(f"  Other image gimbal yaw: {other_gimbal_yaw:.1f}, rotation: {other_rotation:.1f}")
    print(f"  Ref image gimbal yaw: {ref_gimbal_yaw:.1f}, rotation: {ref_rotation:.1f}")
    
    # Extract GPS
    W_other_orig, H_other_orig, lat_other, lon_other, alt_other = extract_exif_gps_alt(other_img_path)
    W_ref_orig, H_ref_orig, lat_ref, lon_ref, alt_ref = extract_exif_gps_alt(ref_img_path)
    
    if None in (lat_other, lon_other, lat_ref, lon_ref):
        print(f"Skipping {pair_name}: missing GPS data")
        return None
    
    # Load images and detections
    other_img = cv2.imread(other_img_path)
    ref_img = cv2.imread(ref_img_path)
    cx_other_orig, cy_other_orig, w_other_orig, h_other_orig = read_yolo_center_px(
        other_lbl_path, W_other_orig, H_other_orig
    )
    
    has_ref_detection = ref_lbl_path is not None
    if has_ref_detection:
        cx_ref_orig, cy_ref_orig, w_ref_orig, h_ref_orig = read_yolo_center_px(
            ref_lbl_path, W_ref_orig, H_ref_orig
        )
    
    # Rotate OTHER image and box to face north
    other_img_rot, cx_other_rot, cy_other_rot, w_other_rot, h_other_rot, W_other_rot, H_other_rot = \
        rotate_image_and_box(other_img, cx_other_orig, cy_other_orig, 
                           w_other_orig, h_other_orig, other_rotation)
    
    # Rotate REF image (and box if exists) to face north
    if has_ref_detection:
        ref_img_rot, cx_ref_rot, cy_ref_rot, w_ref_rot, h_ref_rot, W_ref_rot, H_ref_rot = \
            rotate_image_and_box(ref_img, cx_ref_orig, cy_ref_orig, 
                               w_ref_orig, h_ref_orig, ref_rotation)
    else:
        ref_img_rot, _, _, _, _, W_ref_rot, H_ref_rot = \
            rotate_image_and_box(ref_img, 0, 0, 0, 0, ref_rotation)
        cx_ref_rot = cy_ref_rot = None
    
    # Process GPS projection
    tree_east, tree_north = pixel_to_meters(
        cx_other_rot, cy_other_rot, W_other_rot, H_other_rot, 
        alt_other, FOV, apply_correction=True
    )
    
    tree_lat, tree_lon = meters_to_gps(lat_other, lon_other, tree_east, tree_north)
    
    ref_to_tree_east, ref_to_tree_north = gps_to_meters(lat_ref, lon_ref, tree_lat, tree_lon)
    
    proj_u_ref, proj_v_ref = meters_to_pixel(
        ref_to_tree_east, ref_to_tree_north, W_ref_rot, H_ref_rot, alt_ref, FOV
    )
    
    # Calculate error
    if has_ref_detection:
        pixel_error = math.hypot(proj_u_ref - cx_ref_rot, proj_v_ref - cy_ref_rot)
    else:
        pixel_error = math.hypot(proj_u_ref - W_ref_rot/2, proj_v_ref - H_ref_rot/2)
    
    gps_error = math.hypot(ref_to_tree_east, ref_to_tree_north)
    
    print(f"  Pixel error: {pixel_error:.1f}px")
    print(f"  GPS error: {gps_error:.1f}m")
    
    # Create visualization
    create_visualization(
        pair_name, other_img_rot, ref_img_rot,
        {
            'cx_other': cx_other_rot, 'cy_other': cy_other_rot,
            'w_other': w_other_rot, 'h_other': h_other_rot,
            'proj_u': proj_u_ref, 'proj_v': proj_v_ref,
            'cx_ref': cx_ref_rot, 'cy_ref': cy_ref_rot,
            'W_ref': W_ref_rot, 'H_ref': H_ref_rot,
            'other_rotation': other_rotation,
            'ref_rotation': ref_rotation,
            'pixel_error': pixel_error,
            'gps_error': gps_error
        }
    )
    
    return {
        'pair': pair_name,
        'other_rotation': other_rotation,
        'ref_rotation': ref_rotation,
        'pixel_error': pixel_error,
        'gps_error': gps_error
    }

# ==================== VISUALIZATION ====================
def create_visualization(pair_name, other_img_rot, ref_img_rot, data):
    """Create visualization for gimbal-aligned result."""
    
    # Colors
    GREEN = (0, 255, 0)
    MAGENTA = (255, 0, 255)
    YELLOW = (0, 255, 255)
    WHITE = (255, 255, 255)
    BLUE = (255, 100, 0)
    
    # Draw on OTHER image
    x1 = int(data['cx_other'] - data['w_other']/2)
    y1 = int(data['cy_other'] - data['h_other']/2)
    x2 = int(data['cx_other'] + data['w_other']/2)
    y2 = int(data['cy_other'] + data['h_other']/2)
    cv2.rectangle(other_img_rot, (x1, y1), (x2, y2), GREEN, 3)
    cv2.circle(other_img_rot, (int(data['cx_other']), int(data['cy_other'])), 10, GREEN, -1)
    cv2.putText(other_img_rot, "DETECTED", (x1, max(20, y1-10)), 
               cv2.FONT_HERSHEY_SIMPLEX, 1.0, GREEN, 2)
    cv2.putText(other_img_rot, f"Rotated: {data['other_rotation']:.1f}deg", (20, 40),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, BLUE, 2)
    
    # Draw on REF image
    if 0 <= int(data['proj_u']) < data['W_ref'] and 0 <= int(data['proj_v']) < data['H_ref']:
        cv2.circle(ref_img_rot, (int(data['proj_u']), int(data['proj_v'])), 25, MAGENTA, 3)
        cv2.drawMarker(ref_img_rot, (int(data['proj_u']), int(data['proj_v'])), MAGENTA, 
                      cv2.MARKER_TILTED_CROSS, 35, 3)
        cv2.putText(ref_img_rot, "PROJECTED", (int(data['proj_u'])-50, int(data['proj_v'])+50),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, MAGENTA, 2)
    
    if data['cx_ref'] is not None:
        cv2.circle(ref_img_rot, (int(data['cx_ref']), int(data['cy_ref'])), 20, GREEN, 3)
        cv2.putText(ref_img_rot, "ACTUAL", (int(data['cx_ref'])-40, int(data['cy_ref'])-30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, GREEN, 2)
        if 0 <= int(data['proj_u']) < data['W_ref'] and 0 <= int(data['proj_v']) < data['H_ref']:
            cv2.arrowedLine(ref_img_rot, (int(data['proj_u']), int(data['proj_v'])), 
                          (int(data['cx_ref']), int(data['cy_ref'])), YELLOW, 2, tipLength=0.1)
    
    # Add text info
    cv2.putText(ref_img_rot, f"North-Aligned (rot: {data['ref_rotation']:.1f}deg)", (20, 40),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, BLUE, 2)
    cv2.putText(ref_img_rot, f"Pixel Error: {data['pixel_error']:.1f}px", (20, 80),
               cv2.FONT_HERSHEY_SIMPLEX, 0.9, WHITE, 2)
    cv2.putText(ref_img_rot, f"GPS Error: {data['gps_error']:.1f}m", (20, 120),
               cv2.FONT_HERSHEY_SIMPLEX, 0.9, WHITE, 2)
    
    # Add north arrow on both images
    add_north_arrow(other_img_rot)
    add_north_arrow(ref_img_rot)
    
    # Combine images
    target_h = 800
    scale_other = target_h / other_img_rot.shape[0]
    scale_ref = target_h / ref_img_rot.shape[0]
    other_resized = cv2.resize(other_img_rot, (int(other_img_rot.shape[1] * scale_other), target_h))
    ref_resized = cv2.resize(ref_img_rot, (int(ref_img_rot.shape[1] * scale_ref), target_h))
    combined = np.hstack([other_resized, ref_resized])
    
    output_path = os.path.join(OUTPUT_DIR, f"{pair_name}_gimbal_aligned.jpg")
    cv2.imwrite(output_path, combined)
    print(f"  Saved: {output_path}")

def add_north_arrow(img):
    """Add a north arrow indicator to the image."""
    h, w = img.shape[:2]
    # Position in top-right corner
    cx, cy = w - 60, 60
    length = 40
    
    # Draw arrow pointing up (north)
    cv2.arrowedLine(img, (cx, cy + length//2), (cx, cy - length//2), 
                   (0, 0, 255), 3, tipLength=0.3)
    cv2.putText(img, "N", (cx - 10, cy - length//2 - 10),
               cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

# ==================== MAIN ====================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print("\n" + "="*80)
    print("GPS TREE TRACKER WITH GIMBAL-BASED NORTH ALIGNMENT")
    print("="*80)
    print(f"Pairs directory: {PAIRS_DIR}")
    print(f"Gimbal data file: {GIMBAL_DATA_FILE}")
    print(f"Output directory: {OUTPUT_DIR}")
    print("="*80)
    
    # Parse gimbal data
    print("\nLoading gimbal data...")
    try:
        gimbal_data = parse_gimbal_data(GIMBAL_DATA_FILE)
        print(f"Loaded gimbal data for {len(gimbal_data)} images")
    except Exception as e:
        print(f"Error loading gimbal data: {e}")
        return
    
    # Find all pair folders
    pair_folders = []
    for folder in os.listdir(PAIRS_DIR):
        folder_path = os.path.join(PAIRS_DIR, folder)
        if os.path.isdir(folder_path) and folder.startswith('pair'):
            pair_folders.append(folder_path)
    
    pair_folders.sort()
    print(f"\nFound {len(pair_folders)} pair folders")
    
    # Process each pair
    results = []
    for pair_path in pair_folders:
        result = process_pair_with_gimbal(pair_path, gimbal_data)
        if result:
            results.append(result)
    
    # Summary
    print("\n" + "="*80)
    print("PROCESSING COMPLETE - SUMMARY")
    print("="*80)
    for res in results:
        print(f"{res['pair']}:")
        print(f"  Rotations: other={res['other_rotation']:.1f}, ref={res['ref_rotation']:.1f}")
        print(f"  Errors: pixel={res['pixel_error']:.1f}px, GPS={res['gps_error']:.1f}m")
    
    print(f"\nVisualization results saved to: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()


GPS TREE TRACKER WITH GIMBAL-BASED NORTH ALIGNMENT
Pairs directory: Geolocation/Pairs + EXIF Data
Gimbal data file: Geolocation/Pairs + EXIF Data/GIMBAL_SUMMARY_20251002_212839.txt
Output directory: Geolocation/Results_gimbal_aligned

Loading gimbal data...
Loaded gimbal data for 8 images

Found 4 pair folders

Processing pair1 with gimbal alignment...
  Other image gimbal yaw: 98.6, rotation: -98.6
  Ref image gimbal yaw: 98.6, rotation: -98.6
  Pixel error: 74.4px
  GPS error: 3.5m
  Saved: Geolocation/Results_gimbal_aligned\pair1_gimbal_aligned.jpg

Processing pair2 with gimbal alignment...
  Other image gimbal yaw: -160.6, rotation: 160.6
  Ref image gimbal yaw: -160.6, rotation: 160.6
  Pixel error: 94.0px
  GPS error: 3.2m
  Saved: Geolocation/Results_gimbal_aligned\pair2_gimbal_aligned.jpg

Processing pair3 with gimbal alignment...
  Other image gimbal yaw: -160.6, rotation: 160.6
  Ref image gimbal yaw: -160.6, rotation: 160.6
  Pixel error: 91.6px
  GPS error: 4.5m
  Saved: G